# 13 · Feedback & Corrections
**Use the FeedbackStore to add human corrections and measure the improvement.**

| Step | What happens |
|------|--------------|
| 1 | Load a prior classification run |
| 2 | Add human corrections to the FeedbackStore |
| 3 | Inspect stored examples and selection strategies |
| 4 | Re-classify a sample with feedback injected |
| 5 | Before-vs-after accuracy comparison |

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import asyncio, json
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

# ── Settings ─────────────────────────────────────────────────────────────
DATA_FILE     = "data/Scopus_Content_Collapsed.xlsx"
PRIOR_RUN     = "runs/output/tour_sample_enriched.xlsx"
OUTPUT_FILE   = "runs/output/feedback_sample_enriched.xlsx"
TAXONOMY_DIR  = "generated/taxonomies"
FEEDBACK_DIR  = "runs/feedback"

# Excel file filled in by subject matter experts.
# Columns: uid | transcript | <field> | <field>_corrected | ... | correction_reason
# Any column pair where <field> and <field>_corrected both exist is loaded as a correction.
CORRECTIONS_FILE = "runs/feedback/corrections_examples.xlsx"

ID_COL        = "Reference #"
TEXT_COL      = "Conversation"
SAMPLE_N      = 30
SAMPLE_SEED   = 42

MODEL         = "azure/dsaic-llm-general"
BATCH_SIZE    = 10
CONCURRENCY   = 2
RETRIES       = 3
MAX_TOKENS    = 4096
MODULE        = "contact_reason"

## Step 1 · Load a prior classification run

In [ ]:
prior_path = Path(PRIOR_RUN)
if prior_path.exists():
    df_prior = pd.read_excel(prior_path)
    print(f"Prior run: {len(df_prior)} rows, columns: {list(df_prior.columns)}")

    # Find rows where LLM label differs from ground truth
    label_col = "contact_reason_primary_label"
    gt_col    = "Contact_Reason_Level_1"
    if label_col in df_prior.columns and gt_col in df_prior.columns:
        mismatches = df_prior[df_prior[label_col] != df_prior[gt_col]][[ID_COL, TEXT_COL, gt_col, label_col]]
        print(f"\nMismatches (LLM ≠ ground truth): {len(mismatches)}/{len(df_prior)}")
        display(mismatches.head(10))
    else:
        print(f"Columns {label_col} or {gt_col} not found — run notebook 10 first.")
        df_prior = None
else:
    print(f"Prior run not found at {PRIOR_RUN}. Run notebook 10 first.")
    df_prior = None

## Step 2 · Add corrections to the FeedbackStore
Each correction is a dict with `uid`, `text`, `wrong_label`, `correct_label`, and optional `notes`.

In [ ]:
from rca_pipeline.enhancements.feedback import FeedbackStore

Path(FEEDBACK_DIR).mkdir(parents=True, exist_ok=True)
store = FeedbackStore(FEEDBACK_DIR)
stats = store.get_stats()
existing = stats.get(MODULE, 0)
print(f"FeedbackStore: {existing} existing corrections for '{MODULE}' in {FEEDBACK_DIR}")

In [ ]:
corrections_file = Path(CORRECTIONS_FILE)
if not corrections_file.exists():
    raise FileNotFoundError(
        f"Corrections file not found: {CORRECTIONS_FILE}\n"
        "Ask the subject matter expert to fill in the template and save it there.\n"
        "The template has a 'valid_labels' sheet listing all valid label values."
    )

df_corrections = pd.read_excel(corrections_file)
print(f"Loaded {len(df_corrections)} corrections from {corrections_file.name}")
print(f"Columns: {list(df_corrections.columns)}\n")

# Auto-detect corrected field pairs: any column X where X_corrected also exists
base_cols = [
    c for c in df_corrections.columns
    if f"{c}_corrected" in df_corrections.columns
]
print(f"Corrected fields detected: {base_cols}\n")

# Load each row into FeedbackStore
loaded = 0
skipped = 0
for _, row in df_corrections.iterrows():
    uid      = str(row.get("uid", "")).strip()
    text     = str(row.get("transcript", row.get("text", ""))).strip()
    reason   = str(row.get("correction_reason", row.get("reason", ""))).strip()

    if not uid or not text:
        skipped += 1
        continue

    original  = {col: str(row[col])               for col in base_cols if pd.notna(row[col])}
    corrected = {col: str(row[f"{col}_corrected"]) for col in base_cols if pd.notna(row[f"{col}_corrected"])}

    if not original or not corrected:
        skipped += 1
        continue

    store.add_correction(
        module    = MODULE,
        uid       = uid,
        text      = text,
        original  = original,
        corrected = corrected,
        reason    = reason,
    )
    loaded += 1

total = store.get_stats().get(MODULE, 0)
print(f"Loaded: {loaded}  |  Skipped: {skipped}  |  Total in store: {total}\n")

# Preview
first_col = base_cols[0] if base_cols else ""
print(f"{'UID':<22}  {'Original label':<38}  {'Corrected label':<38}  Reason")
print("-" * 115)
for _, row in df_corrections.iterrows():
    uid    = str(row.get("uid", ""))[:22]
    orig   = str(row.get(first_col, ""))[:38] if first_col else ""
    corr   = str(row.get(f"{first_col}_corrected", ""))[:38] if first_col else ""
    reason = str(row.get("correction_reason", ""))[:35]
    print(f"{uid:<22}  {orig:<38}  {corr:<38}  {reason}")

## Step 3 · Inspect examples and selection strategies
The pipeline supports four strategies for choosing which examples to inject into the prompt.

In [ ]:
corrections = store.load_corrections(MODULE)
print(f"FeedbackStore has {len(corrections)} corrections for '{MODULE}'.\n")

# ── Strategy comparison ───────────────────────────────────────────────────
strategies = {
    "recent":   "Most recently added corrections first. Best when your dataset is "
                "evolving and new error patterns matter more than old ones.",
    "diverse":  "One example per distinct corrected label — maximises taxonomy coverage. "
                "Best default: ensures the LLM sees a spread of correction types, not "
                "five examples all about the same mistake.",
    "frequent": "Most commonly repeated correction pairs first. Best when a small number "
                "of systematic errors dominate and you want the LLM laser-focused on them.",
    "smart":    "Two-stage: scores each correction by quality (has reason + text richness + "
                "recency), then at runtime re-ranks per batch to match examples to the "
                "specific tickets being classified. Highest quality but requires more "
                "corrections in the store to be effective (aim for 20+).",
}

print("=" * 70)
print("  STRATEGY COMPARISON")
print("=" * 70)
for name, description in strategies.items():
    selected = store._select_examples(corrections, 3, name) if name != "smart" else None

    if name == "smart":
        pool = store.build_smart_pool(MODULE)
        print(f"\n  {name.upper():<10}  pool size: {len(pool)} candidates")
    else:
        print(f"\n  {name.upper():<10}  selected: {len(selected) if selected else 0} examples")

    # Wrap description
    words, line = description.split(), ""
    for word in words:
        if len(line) + len(word) + 1 > 60:
            print(f"              {line}")
            line = word
        else:
            line = f"{line} {word}".strip()
    if line:
        print(f"              {line}")

    if selected:
        for ex in selected:
            orig = list(ex.get("original", {}).values())[0] if ex.get("original") else "?"
            corr = list(ex.get("corrected", {}).values())[0] if ex.get("corrected") else "?"
            print(f"              • {str(orig)[:28]:<28} → {str(corr)[:28]}")

print()
print("=" * 70)
print("  RECOMMENDATION")
print("=" * 70)
n = len(corrections)
if n == 0:
    rec = "diverse"
    why = "No corrections yet — add some first."
elif n < 10:
    rec = "diverse"
    why = f"Only {n} corrections. Diverse maximises coverage across the few you have."
elif n < 20:
    rec = "frequent"
    why = f"{n} corrections. Check for repeated errors — frequent will target them."
else:
    rec = "smart"
    why = f"{n} corrections — enough for smart's two-stage scoring to be effective."

print(f"\n  With {n} correction(s): use  '{rec}'")
print(f"  {why}")

In [ ]:
# Full dump of stored corrections
all_corrections = store.load_corrections(MODULE)
if all_corrections:
    rows = []
    for c in all_corrections:
        orig = list(c.get("original", {}).values())
        corr = list(c.get("corrected", {}).values())
        rows.append({
            "uid":       c.get("uid", ""),
            "original":  orig[0] if orig else "",
            "corrected": corr[0] if corr else "",
            "reason":    c.get("reason", ""),
            "timestamp": c.get("timestamp", ""),
        })
    display(pd.DataFrame(rows))
else:
    print(f"No corrections stored yet for module '{MODULE}'.")

## Step 4 · Re-classify with feedback injected
The pipeline automatically injects selected examples into each classification prompt.

In [ ]:
import time
from rca_pipeline.config import RunConfig
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import drop_enrichment_cols
from rca_pipeline.pipeline import prepare_batched_records, run_classification_module, MODULE_REGISTRY
from rca_pipeline.llm_client import set_system_prompt_suffix, clear_system_prompt_suffix

df_full = pd.read_excel(DATA_FILE)
df = df_full.sample(n=min(SAMPLE_N, len(df_full)), random_state=SAMPLE_SEED).reset_index(drop=True)
df[ID_COL] = df[ID_COL].astype(str)

Path("runs/output").mkdir(parents=True, exist_ok=True)
_tmp = "runs/output/_feedback_tmp.xlsx"
df.to_excel(_tmp, index=False)

cfg = RunConfig(
    input_file   = _tmp,
    output_file  = OUTPUT_FILE,
    taxonomy_dir = TAXONOMY_DIR,
    id_col       = ID_COL,
    text_col     = TEXT_COL,
    modules      = [MODULE],
    model        = MODEL,
    batch_size   = BATCH_SIZE,
    concurrency  = CONCURRENCY,
    retries      = RETRIES,
    max_tokens   = MAX_TOKENS,
    feedback_dir          = FEEDBACK_DIR,
    feedback_max_examples = 5,
    feedback_strategy     = "diverse",
)

taxonomies = load_taxonomies(TAXONOMY_DIR)
df_work    = drop_enrichment_cols(df.copy())
batches    = prepare_batched_records(df_work, cfg)

# Build and inject the few-shot prompt suffix
few_shot = store.build_few_shot_prompt(MODULE, max_examples=5, selection_strategy="diverse")
if few_shot:
    set_system_prompt_suffix(few_shot)
    print(f"Injecting {len(store.load_corrections(MODULE))} corrections as few-shot examples.")
else:
    print("No corrections available — classifying without feedback.")

results_with_fb = []

async def _classify_fb():
    if MODULE not in MODULE_REGISTRY:
        print(f"Module {MODULE} not in registry")
        return
    t0 = time.time()
    res = await run_classification_module(MODULE, batches, taxonomies, cfg)
    results_with_fb.extend(res)
    print(f"  {MODULE}: {len(res)} records  {time.time()-t0:.1f}s")

print(f"\nClassifying {len(df_work)} rows...")
asyncio.run(_classify_fb())
clear_system_prompt_suffix()
print(f"Done — {len(results_with_fb)} results")

## Step 5 · Before-vs-after comparison

In [ ]:
from rca_pipeline.utils import post_process, write_output_file

grouped = {}
for r in results_with_fb:
    uid = str(r.get("uid", ""))
    grouped.setdefault(uid, {"uid": uid}).update(r)

df_fb_out = post_process(df_work, list(grouped.values()), taxonomies, cfg)
write_output_file(df_fb_out, OUTPUT_FILE)
print(f"Output: {len(df_fb_out)} rows × {len(df_fb_out.columns)} columns → {OUTPUT_FILE}\n")

In [ ]:
label_col = "contact_reason_primary_label"
code_col  = "contact_reason_primary_code"

# ── 1. Label distribution: with vs without feedback ───────────────────────
print("=" * 60)
print("  LABEL DISTRIBUTION — with feedback")
print("=" * 60)
if label_col in df_fb_out.columns:
    dist = df_fb_out[label_col].value_counts()
    for label, count in dist.items():
        bar = "█" * count
        print(f"  {label[:38]:<38}  {count:>3}  {bar}")
else:
    print(f"  Column {label_col!r} not found.")
print()

# ── 2. Label distribution from prior run (without feedback) ───────────────
if df_prior is not None and label_col in df_prior.columns:
    # Align to same rows
    shared_ids = set(df_fb_out[ID_COL].astype(str)) & set(df_prior[ID_COL].astype(str))
    prior_sub  = df_prior[df_prior[ID_COL].astype(str).isin(shared_ids)]
    after_sub  = df_fb_out[df_fb_out[ID_COL].astype(str).isin(shared_ids)]

    print("=" * 60)
    print(f"  LABEL CHANGES  (comparing {len(shared_ids)} shared rows)")
    print("=" * 60)

    prior_labels = prior_sub.set_index(ID_COL)[label_col].rename("before")
    after_labels = after_sub.set_index(ID_COL)[label_col].rename("after")
    comparison   = pd.concat([prior_labels, after_labels], axis=1, join="inner")
    changed      = comparison[comparison["before"] != comparison["after"]]

    print(f"  Rows where label changed : {len(changed)}/{len(comparison)}")
    print()
    if len(changed) > 0:
        for uid, row in changed.iterrows():
            print(f"  [{uid}]")
            print(f"    before : {row['before']}")
            print(f"    after  : {row['after']}")
            print()
    else:
        print("  No label changes — the few-shot examples did not alter predictions")
        print("  on this sample. This is normal with only 5 synthetic examples.")
        print("  More corrections covering common error patterns will increase impact.")
else:
    print("  No prior run available for comparison (run notebook 10 first).")
print()

# ── 3. Verify the few-shot prompt was injected ────────────────────────────
print("=" * 60)
print("  FEEDBACK INJECTION SUMMARY")
print("=" * 60)
corrections_in_store = store.load_corrections(MODULE)
print(f"  Corrections in store : {len(corrections_in_store)}")
print(f"  Injected as few-shot : up to 5 (strategy='diverse')")
sample_prompt = store.build_few_shot_prompt(MODULE, max_examples=5, selection_strategy="diverse")
print(f"  Prompt block size    : {len(sample_prompt):,} chars")
if sample_prompt:
    print()
    print("  First 300 chars of injected block:")
    print("  " + sample_prompt[:300].replace("\n", "\n  "))
print("=" * 60)